# EBI BioImage Archive — add new folders (TetraSpeck beads; earlier: QDots MixD, Revision 2)

**Dataset:** 2026 Multicolour Bayer-SMLM paper (accession `S-BIAD3210`)
**Source:** SMB share, mounted at `/scratch/sycamore-asap/` on this machine
**Destination:** `ftp-private.ebi.ac.uk` (BioImage Archive Webin FTP)

## What this adds

- `Ximea/TetraspeckBeads` — the bead acquisitions behind Figure 2d (spatial and spectral precision
  vs photons): 488, 561 and 638 nm excitation at 1–10 mW, 25 positions each. Only part of the
  1 mW, 488 nm set (one TIFF and its 25 metadata files, plus one 10 mW TIFF) was deposited before.
  Copy the rest into `Data/Ximea/TetraspeckBeads/` on the share first, keeping the existing
  `Tetraspeck_<laser>_excitation/<power> excitation/` layout.

Earlier runs of this notebook added `Ximea/QDots/MixD` (the QDot mixture for the SI "Example_Fitting"
panels) and `Ximea/Revision_2` (data acquired for the second revision). Listing a folder again is
harmless: only files missing from the manifest are hashed, and the upload skips anything already
on EBI at the right size.

For each folder: every raw TIFF **and** every MicroManager `*metadata.txt` sidecar (the camera
ROI the fitting code needs — see `EBI_Upload_Metadata.ipynb`). Other file types are listed but
not uploaded, so the deposition stays raw data + metadata.

## What this notebook does, in order

1. **Load the manifest** `2026_Multicolour_ebi_filelist.tsv`.
2. **Scan the new folders** (dry run): files to add, files skipped, and any TIFF without a
   matching metadata file. Nothing is written or uploaded in this step.
3. **Hash** the new files.
4. **Merge** them into the manifest (backed up as `.bak` first).
5. **Upload** (size-based resume; safe to re-run).
6. **Verify** every new file is on the remote at the right size.
7. **Rebuild + re-upload** `biostudies_checksums.json`.
8. **Rebuild** `2026_Multicolour_Filelist.json` — submit it through the portal by hand.

Run this and `EBI_Upload_Metadata.ipynb` one at a time, not concurrently: both rewrite the manifest.
Either order works — each starts from the manifest as it currently is.

## Credentials

Read from a local credentials file — by default `~/.config/pyS3M/ebi_ftp_credentials.txt`, or
wherever `EBI_CREDENTIALS_FILE` points (see `_ebi_credentials.py`). Never stored here.

---
## 0 — Imports

In [ ]:
import csv
import ftplib
import hashlib
import json
import re
import shutil
import sys
import time
from collections import Counter
from datetime import timedelta
from pathlib import Path

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not available — progress will be printed every 10 files")

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
# ── Source data (share mounted at /scratch/ on this machine) ───────────────────
SMB_ROOT = Path('/scratch/sycamore-asap/2026_Multicolour_Paper/Data')
OUT_DIR  = Path('/scratch/sycamore-asap/2026_Multicolour_Paper/EBI_Upload')

# ── Folders to add, relative to SMB_ROOT ──────────────────────────────────────
NEW_FOLDERS = [
    "Ximea/TetraspeckBeads",   # Figure 2d; earlier runs: "Ximea/QDots/MixD", "Ximea/Revision_2"
]

# ── Manifest = what is on EBI; extended in place (backed up first) ────────────
BASELINE_TSV = OUT_DIR / "2026_Multicolour_ebi_filelist.tsv"

# ── Session outputs ────────────────────────────────────────────────────────────
NEW_UPLOAD_PAIRS = OUT_DIR / "pending_new_folders_3_upload_paths.txt"
CHECKSUMS_JSON   = OUT_DIR / "biostudies_checksums.json"
FILELIST_JSON    = OUT_DIR / "2026_Multicolour_Filelist.json"

# ── EBI path prefix (preserved inside the submission) ─────────────────────────
EBI_PREFIX = "2026_Multicolour_Paper"
ACCESSION  = "S-BIAD3210"

# ── What gets deposited: raw TIFFs + MicroManager metadata sidecars ──────────
TIFF_RE         = re.compile(r"\.(ome\.)?tiff?$", re.IGNORECASE)
METADATA_SUFFIX = "metadata.txt"

# ── FTP credentials (EBI-issued) ────────────────────────────────────────────
# Read from a local credentials file -- never stored in this git-tracked notebook.
# See _ebi_credentials.py for where the file lives and its format.
from _ebi_credentials import load_ebi_credentials
_creds   = load_ebi_credentials()
FTP_HOST = "ftp-private.ebi.ac.uk"
FTP_USER = _creds["WEBIN_USER"]
FTP_PASS = _creds["WEBIN_PASS"]
FTP_ROOT = _creds["EBI_FTP_DIR"]   # upload root: the submission's private FTP directory

# ── Sanity check ──────────────────────────────────────────────────────────────
for label, p in [("Data folder", SMB_ROOT), ("Manifest", BASELINE_TSV)]:
    print(f"{label:<12}: {p}  {'OK' if p.exists() else 'NOT FOUND — check the /scratch/ mount'}")
for sub in NEW_FOLDERS:
    folder = SMB_ROOT / sub
    print(f"New folder  : {sub}  {'OK' if folder.is_dir() else 'NOT FOUND'}")
    if not folder.is_dir() and folder.parent.is_dir():
        print(f"              {folder.parent.relative_to(SMB_ROOT)}/ contains: "
              f"{sorted(p.name for p in folder.parent.iterdir() if p.is_dir())}")
print(f"FTP host    : {FTP_HOST}")
print(f"FTP user    : {FTP_USER}")

---
## Helper functions

Same hashing, manifest and FTP plumbing as `EBI_Upload_Metadata.ipynb`.

In [ ]:
def md5sum(path: Path, chunk: int = 1 << 20) -> str:
    """Return hex MD5 of a file, reading in 1 MB chunks."""
    h = hashlib.md5()
    with path.open("rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()


def human_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"


def load_manifest(path: Path) -> list:
    """Read the TSV manifest into a list of {filename, md5, size} dicts."""
    rows = []
    with path.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        name_key = "filename" if "filename" in (reader.fieldnames or []) else "files"
        for row in reader:
            rows.append({"filename": row[name_key], "md5": row["md5"], "size": int(row["size"])})
    return rows


def write_manifest(rows: list, out_path: Path) -> None:
    """Write {filename, md5, size} rows to a TSV manifest, sorted by filename."""
    with out_path.open("w") as f:
        f.write("filename\tmd5\tsize\n")
        for r in sorted(rows, key=lambda r: r["filename"]):
            f.write(f"{r['filename']}\t{r['md5']}\t{r['size']}\n")


def is_deposited_type(p: Path) -> bool:
    return bool(TIFF_RE.search(p.name)) or p.name.endswith(METADATA_SUFFIX)


def is_covered(tiff_name: str, meta_stems: set) -> bool:
    """True if a '<stem>_metadata.txt' belongs to this TIFF: the stem is the TIFF name without
    its extension, or that name minus a trailing _1, _2 ... continuation-file suffix. Grid
    positions ('..._MMStack_2-Pos000_001') keep their own stem."""
    name = TIFF_RE.sub("", tiff_name)
    base = re.sub(r"_\d+$", "", name)
    return name in meta_stems or base in meta_stems


def scan_new_folders(smb_root: Path, subfolders: list) -> tuple:
    """Return (files to deposit, skipped other files, TIFFs without metadata) under subfolders."""
    keep, skipped, uncovered = [], [], []
    for sub in subfolders:
        root = smb_root / sub
        if not root.is_dir():
            print(f"WARNING: folder not found, skipping: {root}")
            continue
        for p in sorted(root.rglob("*")):
            if p.is_file():
                (keep if is_deposited_type(p) else skipped).append(p)
    by_dir = {}
    for p in keep:
        by_dir.setdefault(p.parent, []).append(p)
    for folder, files in by_dir.items():
        meta_stems = {f.name[: -len(METADATA_SUFFIX)].rstrip("_") for f in files if f.name.endswith(METADATA_SUFFIX)}
        uncovered += [f for f in files if TIFF_RE.search(f.name) and not is_covered(f.name, meta_stems)]
    return keep, skipped, uncovered


def remote_size(ftp: ftplib.FTP, path: str) -> int:
    """Return remote file size, or -1 if the file does not exist."""
    try:
        return ftp.size(path)
    except ftplib.error_perm:
        return -1


def ensure_remote_dirs(ftp: ftplib.FTP, remote_path: str):
    """Recursively create all intermediate directories on the FTP server."""
    parts = Path(remote_path).parent.parts
    for i in range(1, len(parts) + 1):
        try:
            ftp.mkd(str(Path(*parts[:i])))
        except ftplib.error_perm:
            pass  # already exists — fine


def ftp_connect() -> ftplib.FTP:
    ftp = ftplib.FTP(FTP_HOST, timeout=60)
    ftp.login(FTP_USER, FTP_PASS)
    ftp.set_pasv(True)
    return ftp


def upload_file(ftp: ftplib.FTP, local: Path, remote: str, chunk: int = 1 << 20) -> str:
    """Upload local → remote, skipping if sizes match. Returns 'skip'/'ok'."""
    if remote_size(ftp, remote) == local.stat().st_size:
        return "skip"
    ensure_remote_dirs(ftp, remote)
    with local.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=chunk)
    return "ok"


def run_upload(pairs: list) -> dict:
    """Upload (local Path, manifest-relative remote path) pairs, reconnecting after errors."""
    n = len(pairs)
    print(f"Uploading {n} files to {FTP_HOST} …")
    ftp = ftp_connect()
    counts = {"ok": 0, "skip": 0, "error": 0}
    t0 = time.time()
    iterator = tqdm(enumerate(pairs, 1), total=n, unit="file") if HAS_TQDM else enumerate(pairs, 1)
    for i, (local, rel_remote) in iterator:
        try:
            status = upload_file(ftp, local, f"{FTP_ROOT}/{rel_remote}")
        except Exception as exc:
            status = "error"
            print(f"  ERROR [{i}/{n}] {local.name}: {exc}", file=sys.stderr)
            try:
                ftp.quit()
            except Exception:
                pass
            time.sleep(5)
            ftp = ftp_connect()
        counts[status] += 1
        if not HAS_TQDM and i % 10 == 0:
            print(f"  [{i}/{n}]  ok={counts['ok']}  skip={counts['skip']}  err={counts['error']}")
    try:
        ftp.quit()
    except Exception:
        pass
    print(f"\nFinished in {timedelta(seconds=int(time.time() - t0))}: "
          f"uploaded {counts['ok']}, skipped {counts['skip']}, errors {counts['error']}")
    return counts

---
## 1 — Load the manifest (what is on EBI)

Read-only.

In [ ]:
manifest_rows = load_manifest(BASELINE_TSV)
manifest_paths = {r["filename"] for r in manifest_rows}
print(f"Manifest files : {len(manifest_rows):,} ({human_size(sum(r['size'] for r in manifest_rows))})")

---
## 2 — Scan the new folders (dry run — nothing written or uploaded)

**Check this output before going on:** the skipped file types (anything that isn't a TIFF or a
`*metadata.txt`), and any TIFF with no matching metadata file.

In [ ]:
deposit, skipped, uncovered = scan_new_folders(SMB_ROOT, NEW_FOLDERS)
new_files = [p for p in deposit if f"{EBI_PREFIX}/{p.relative_to(SMB_ROOT)}" not in manifest_paths]

print(f"Files to deposit   : {len(deposit):,} ({human_size(sum(p.stat().st_size for p in deposit))})")
print(f"  of which new     : {len(new_files):,} ({human_size(sum(p.stat().st_size for p in new_files))})")
print(f"  TIFFs / metadata : {sum(bool(TIFF_RE.search(p.name)) for p in deposit)} / "
      f"{sum(p.name.endswith(METADATA_SUFFIX) for p in deposit)}")
print()
print(f"{'Folder':<90} {'files':>6} {'size':>10}")
by_folder = Counter()
size_by_folder = Counter()
for p in deposit:
    key = str(p.parent.relative_to(SMB_ROOT))
    by_folder[key] += 1
    size_by_folder[key] += p.stat().st_size
for key in sorted(by_folder):
    print(f"{key:<90} {by_folder[key]:>6} {human_size(size_by_folder[key]):>10}")

print()
if skipped:
    kinds = Counter(p.suffix.lower() or p.name for p in skipped)
    print(f"SKIPPED (not TIFF / metadata): {len(skipped)} file(s) — {dict(kinds)}")
    for p in skipped[:20]:
        print(" ", p.relative_to(SMB_ROOT))
    if len(skipped) > 20:
        print(f"  ... and {len(skipped) - 20} more")
else:
    print("SKIPPED: none")

print()
if uncovered:
    print(f"COVERAGE: {len(uncovered)} TIFF(s) with no matching *_metadata.txt:")
    for p in uncovered:
        print(" ", p.relative_to(SMB_ROOT))
else:
    print("COVERAGE: every TIFF has a matching metadata file.")

---
## 3 — Hash the new files

These are large multi-GB OME-TIFs over SMB — this can take hours; run it in `tmux`/`screen`
or leave the notebook open.

In [ ]:
new_rows, pairs = [], []
t0 = time.time()
iterator = tqdm(new_files, unit="file") if HAS_TQDM else new_files
for i, src in enumerate(iterator, 1):
    ebi_path = f"{EBI_PREFIX}/{src.relative_to(SMB_ROOT)}"
    new_rows.append({"filename": ebi_path, "md5": md5sum(src), "size": src.stat().st_size})
    pairs.append((src, ebi_path))
    if not HAS_TQDM and i % 10 == 0:
        print(f"  {i}/{len(new_files)}  elapsed {timedelta(seconds=int(time.time() - t0))}")

with NEW_UPLOAD_PAIRS.open("w") as pp:
    for src, ebi_path in pairs:
        pp.write(f"{src}\t{ebi_path}\n")
print(f"Hashed {len(new_rows)} file(s) in {timedelta(seconds=int(time.time() - t0))}. Upload pairs: {NEW_UPLOAD_PAIRS}")

---
## 4 — Merge into the manifest

Backs up `2026_Multicolour_ebi_filelist.tsv` as `.bak`, then rewrites it with the new rows added.

In [ ]:
if new_rows:
    backup_path = BASELINE_TSV.with_suffix(BASELINE_TSV.suffix + ".bak")
    # copyfile, not copy/copy2: this SMB mount rejects the chmod/utime those try afterwards.
    shutil.copyfile(BASELINE_TSV, backup_path)
    print(f"Backed up manifest to {backup_path}")
    merged_rows = manifest_rows + new_rows
    write_manifest(merged_rows, BASELINE_TSV)
    print(f"Merged manifest : {len(merged_rows):,} files ({human_size(sum(r['size'] for r in merged_rows))})")
else:
    merged_rows = manifest_rows
    print("No new rows — manifest left unchanged.")

---
## 5 — Upload the new folders (FTP)

Size-based resume: re-running skips anything already uploaded at the correct size. Uploads every
file found in §2 (not only the newly hashed ones), so a re-run also finishes an upload
interrupted in an earlier session.

In [ ]:
all_pairs = [(p, f"{EBI_PREFIX}/{p.relative_to(SMB_ROOT)}") for p in deposit]
counts = run_upload(all_pairs)

---
## 6 — Verify every new file is on the remote, at the right size

In [ ]:
ftp = ftp_connect()
wrong = []
for local, rel_remote in all_pairs:
    if remote_size(ftp, f"{FTP_ROOT}/{rel_remote}") != local.stat().st_size:
        wrong.append(rel_remote)
ftp.quit()
if wrong:
    print(f"MISMATCH — {len(wrong)} file(s) missing or the wrong size on the remote:")
    for w in wrong[:20]:
        print(" ", w)
    print("Re-run §5.")
else:
    print(f"MATCH — all {len(all_pairs)} files present on the remote at the correct size.")

---
## 7 — Rebuild the checksums manifest and re-upload it

In [ ]:
def build_checksums_json(rows: list, out_path: Path, accession: str) -> dict:
    """Build biostudies_checksums.json ({path: {md5, size}}) from manifest rows."""
    files = {r["filename"]: {"md5": r["md5"], "size": r["size"]} for r in rows}
    manifest = {
        "_meta": {
            "description": (
                "MD5 checksums for the raw acquisition files (and their MicroManager "
                f"metadata.txt sidecars) deposited to BioStudies {accession} (2026 Multicolour "
                "Bayer-SMLM paper) -- a curated subset of the full acquired dataset. Computed "
                "from the original local files before upload "
                "(notebooks/EBI_Upload/EBI_Upload_Notebook.ipynb, EBI_Upload_Update.ipynb, "
                "EBI_Upload_Metadata.ipynb, EBI_Upload_NewFolders.ipynb), so a mismatch after "
                "download indicates real corruption/truncation."
            ),
            "accession": accession,
            "algorithm": "md5",
            "n_files": len(files),
            "total_bytes": sum(r["size"] for r in rows),
        },
        "files": files,
    }
    out_path.write_text(json.dumps(manifest, indent=2))
    print(f"Wrote {out_path}  ({len(files):,} files)")
    return manifest


build_checksums_json(merged_rows, CHECKSUMS_JSON, ACCESSION)

remote = f"{FTP_ROOT}/{EBI_PREFIX}/Checksums/{CHECKSUMS_JSON.name}"
ftp = ftp_connect()
ensure_remote_dirs(ftp, remote)
with CHECKSUMS_JSON.open("rb") as f:
    ftp.storbinary(f"STOR {remote}", f, blocksize=1 << 20)
ok = remote_size(ftp, remote) == CHECKSUMS_JSON.stat().st_size
ftp.quit()
print("MATCH — checksums manifest uploaded." if ok else "MISMATCH — re-run this cell.")

---
## 8 — Rebuild the EBI file-list JSON (for the portal)

**Not** uploaded automatically — submit `2026_Multicolour_Filelist.json` through the BioImage
Archive portal by hand, or the new files won't appear in the public study.

In [ ]:
def build_ebi_filelist_json(rows: list, checksums_path: Path, ebi_prefix: str, out_path: Path) -> list:
    """Flat BioStudies file-listing JSON from manifest rows, plus the checksums manifest."""
    entries = [
        {"path": r["filename"], "size": r["size"],
         "attributes": [{"name": "size", "value": str(r["size"])}], "type": "file"}
        for r in sorted(rows, key=lambda r: r["filename"])
    ]
    cksum_size = checksums_path.stat().st_size
    entries.append({
        "path": f"{ebi_prefix}/Checksums/{checksums_path.name}", "size": cksum_size,
        "attributes": [{"name": "size", "value": str(cksum_size)}], "type": "file",
    })
    out_path.write_text(json.dumps(entries, indent=2))
    print(f"Wrote {out_path}  ({len(entries):,} entries)")
    return entries


build_ebi_filelist_json(merged_rows, CHECKSUMS_JSON, EBI_PREFIX, FILELIST_JSON)

---
## Checklist

- [ ] Running on a machine with the share mounted at `/scratch/sycamore-asap/...`
- [ ] Credentials file copied to this machine (`~/.config/pyS3M/ebi_ftp_credentials.txt`, `chmod 600`)
- [ ] Not running `EBI_Upload_Metadata.ipynb` at the same time (both rewrite the manifest)
- [ ] Remaining bead data copied to `Data/Ximea/TetraspeckBeads/` on the share
- [ ] §1: the new folder found (`Ximea/TetraspeckBeads`)
- [ ] §2 reviewed — skipped file types and metadata coverage as expected
- [ ] Hashed (§3) and merged into the manifest, `.bak` written (§4)
- [ ] Uploaded (§5), errors = 0, verified on the remote (§6)
- [ ] `biostudies_checksums.json` rebuilt and re-uploaded (§7)
- [ ] `2026_Multicolour_Filelist.json` rebuilt (§8) **and submitted through the BioImage Archive portal**